# Task 7: Text Cleaning

Clean the standardized WELFake `title` and `text` fields by removing punctuation, digits, special characters, and non-alphabetic symbols while preserving words and spaces.

## Scope

This notebook handles missing values, lowercases text, removes characters outside `a-z` and whitespace, collapses repeated whitespace, validates the cleaned columns, and saves the result. It does not perform stemming, lemmatization, stopword removal, TF-IDF, or model training.

In [1]:
from pathlib import Path
import re

import pandas as pd

In [2]:
def to_safe_text(value):
    """Convert missing or non-string values into a clean string."""
    if pd.isna(value):
        return ""
    return str(value).strip()


def clean_text(value):
    """Remove punctuation, numbers, and non-alphabetic symbols."""
    text = to_safe_text(value).lower()
    if not text:
        return ""

    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def verify_cleaned(df, columns):
    """Count values containing characters outside letters and spaces."""
    issues = []
    for column in columns:
        bad_rows = df[column].apply(
            lambda value: bool(re.search(r"[^a-z\s]", value))
        )
        issues.append((column, int(bad_rows.sum())))
    return issues

In [3]:
project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

input_path = project_root / "data" / "data_cleaned_task6.csv"
output_path = project_root / "data" / "data_cleaned_task7.csv"
text_columns = ["title", "text"]

df = pd.read_csv(input_path)

print("Input dataset:")
print(f"Rows: {len(df)} | Columns: {list(df.columns)}")
print("Sample before cleaning:")
display(df[text_columns].head(3))

before_examples = df[text_columns].head(3).copy()

Input dataset:
Rows: 63172 | Columns: ['title', 'text', 'label', 'title_norm', 'text_norm', 'combined_norm']
Sample before cleaning:


,title,text
0,law enforcement on high alert following threat...,no comment is expected from barack obama membe...
1,NaN,did they post their votes for hillary already?
2,unbelievable! obama’s attorney general says mo...,"now, most of the demonstrators gathered last n..."


In [4]:
for column in text_columns:
    df[column] = df[column].apply(clean_text)

cleaned_issues = verify_cleaned(df, text_columns)
print("Validation check after cleaning:")
for column, bad_rows in cleaned_issues:
    print(f"{column}: remaining invalid characters = {bad_rows}")

print("Sample after cleaning:")
display(df[text_columns].head(3))

Validation check after cleaning:
title: remaining invalid characters = 0
text: remaining invalid characters = 0
Sample after cleaning:


,title,text
0,law enforcement on high alert following threat...,no comment is expected from barack obama membe...
1,,did they post their votes for hillary already
2,unbelievable obama s attorney general says mos...,now most of the demonstrators gathered last ni...


In [5]:
print("Before/after examples:")
for index in range(3):
    print(f"\nExample {index + 1}")
    print(f"Before title : {before_examples.iloc[index]['title']}")
    print(f"After title  : {df.iloc[index]['title']}")
    print(f"Before text  : {before_examples.iloc[index]['text']}")
    print(f"After text   : {df.iloc[index]['text']}")

Before/after examples:

Example 1
Before title : law enforcement on high alert following threats against cops and whites on 9-11by #blacklivesmatter and #fyf911 terrorists [video]
After title  : law enforcement on high alert following threats against cops and whites on by blacklivesmatter and fyf terrorists video
Before text  : no comment is expected from barack obama members of the #fyf911 or #fukyoflag and #blacklivesmatter movements called for the lynching and hanging of white people and cops. they encouraged others on a radio show tuesday night to turn the tide and kill white people and cops to send a message about the killing of black people in america.one of the f***yoflag organizers is called sunshine. she has a radio blog show hosted from texas called, sunshine s f***ing opinion radio show. a snapshot of her #fyf911 @lolatwhitefear twitter page at 9:53 p.m. shows that she was urging supporters to call now!! #fyf911 tonight we continue to dismantle the illusion of white below is

In [6]:
df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to: {output_path}")
print("Conclusion: The standardized WELFake text columns were cleaned by removing punctuation, digits, and non-alphanumeric symbols while preserving word boundaries and spaces. Missing and non-string values were safely converted to empty strings, and the cleaned text is ready for later text-processing stages without stemming, lemmatization, or stopword removal.")

Saved cleaned dataset to: c:\Users\ADMIN\Downloads\Task-06\data\data_cleaned_task7.csv
Conclusion: The standardized WELFake text columns were cleaned by removing punctuation, digits, and non-alphanumeric symbols while preserving word boundaries and spaces. Missing and non-string values were safely converted to empty strings, and the cleaned text is ready for later text-processing stages without stemming, lemmatization, or stopword removal.
